# `_chunk_scan_fwd` in JAX Pallas

## What this kernel computes

Given:
- `CB  (batch, nchunks, ngroups, chunk_size, chunk_size)` — pre-computed Gram matrix C@B^T
- `x   (batch, seqlen, nheads, hdim)` — SSM input
- `dt  (batch, nheads, nchunks, chunk_size)` — discretization step
- `dA  (batch, nheads, nchunks, chunk_size)` — cumulative log-decay (dA_cumsum)
- `C   (batch, seqlen, ngroups, dstate)` — output-projection vectors
- `states (batch, nchunks, nheads, hdim, dstate)` — per-chunk states (from `_state_passing_fwd`)

It produces `Y (batch, seqlen, nheads, hdim)` as the sum of two terms:

```
Y[b, c*Q+m, h, :] =

  Y_off:  exp(dA[b,h,c,m]) * C[b,c*Q+m, g, :] @ states[b,c, h, :, :]^T
          (inter-chunk: contribution of all previous chunks via propagated state)

+ Y_diag: sum_{k<=m}  CB[b,c,g,m,k]                          -- C@B^T at (m,k)
                     * exp(min(dA[b,h,c,m] - dA[b,h,c,k], 0)) -- intra-chunk decay m→k
                     * dt[b,h,c,k]                              -- ZOH discretization
                     * x[b, c*Q+k, h, :]                        -- input at position k
          (intra-chunk: lower-triangular scan, causal)
```

where `g = h // ratio` (GQA: `ratio = nheads // ngroups` heads share one B/C group).

## Connection to `ssd_minimal.py`

Yes — this kernel computes **exactly** `Y_diag + Y_off` from `ssd_minimal_discrete`:

| minimal.py | this kernel | note |
|---|---|---|
| `Y_off = einsum('bclhn,bchpn,bhcl->bclhp', C, states, exp(dA_cs))` | same | inter-chunk C@state |
| `Y_diag = einsum("bclhn,bcshn,bhcls,bcshp->bclhp", C, B, L, X)` | same | intra-chunk with L=CB*decay |

The difference is that this kernel takes **pre-computed** `CB = C@B^T` (from `_bmm_chunk_fwd`)
rather than computing the inner product inline.

## Pipeline position
```
_chunk_cumsum_fwd  →  dA_cumsum
_chunk_state_fwd   →  raw states
_state_passing_fwd →  propagated states    ← states input here
_bmm_chunk_fwd     →  CB                   ← CB input here
_chunk_scan_fwd    →  Y                    ← THIS KERNEL
```

In [1]:
import os, sys, types, importlib.util, time
import numpy as np

os.environ.setdefault(
    "CC",
    os.path.expanduser("~/miniconda3/envs/cutedsl/bin/x86_64-conda-linux-gnu-gcc"),
)

import jax
import jax.numpy as jnp
from jax import lax
import jax.experimental.pallas as pl
from jax._src.pallas.triton.core import CompilerParams

# ── load mamba modules via importlib (avoids __init__.py CUDA deps) ──────────
MAMBA_ROOT = os.path.expanduser("~/mamba")

def _stub_pkg(name, path=None):
    """Create a minimal package stub so sub-modules can set themselves up."""
    m = types.ModuleType(name)
    if path:
        m.__path__ = [path]
    m.__package__ = name
    sys.modules[name] = m
    return m

def _load_mod(name, relpath):
    """Load a real module from MAMBA_ROOT/relpath, registering it in sys.modules."""
    spec = importlib.util.spec_from_file_location(
        name, os.path.join(MAMBA_ROOT, relpath)
    )
    m = importlib.util.module_from_spec(spec)
    sys.modules[name] = m
    spec.loader.exec_module(m)
    return m

# Stub package hierarchy so relative imports resolve
_stub_pkg("mamba_ssm",            os.path.join(MAMBA_ROOT, "mamba_ssm"))
_stub_pkg("mamba_ssm.utils",      os.path.join(MAMBA_ROOT, "mamba_ssm/utils"))
_stub_pkg("mamba_ssm.ops",        os.path.join(MAMBA_ROOT, "mamba_ssm/ops"))
_stub_pkg("mamba_ssm.ops.triton", os.path.join(MAMBA_ROOT, "mamba_ssm/ops/triton"))

# Load real modules in dependency order
_load_mod("mamba_ssm.utils.determinism",    "mamba_ssm/utils/determinism.py")
_load_mod("mamba_ssm.ops.triton.ssd_bmm",   "mamba_ssm/ops/triton/ssd_bmm.py")
scan_mod = _load_mod("mamba_ssm.ops.triton.ssd_chunk_scan",
                     "mamba_ssm/ops/triton/ssd_chunk_scan.py")

import torch
from mamba_ssm.ops.triton.ssd_chunk_scan import _chunk_scan_fwd
from mamba_ssm.ops.triton.ssd_bmm import _bmm_chunk_fwd

def to_torch_bf16(x):
    """JAX bfloat16 → CUDA torch bfloat16 (via float32 to avoid ml_dtypes issue)."""
    return torch.from_numpy(np.array(x.astype(jnp.float32))).bfloat16().cuda()

def to_torch_f32(x):
    """JAX float32 → CUDA torch float32."""
    return torch.from_numpy(np.array(x.astype(jnp.float32))).cuda()

print(f"JAX     : {jax.__version__}")
print(f"devices : {jax.devices()}")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")

JAX     : 0.9.0.1
devices : [CudaDevice(id=0)]
PyTorch : 2.8.0+cu129
GPU     : NVIDIA GeForce RTX 4090


## Triton Reference: `_chunk_scan_fwd_kernel`

### Grid
```
grid = (
    cdiv(chunk_size, BLOCK_M) * cdiv(hdim, BLOCK_N),   # axis-0: output tiles (m, n)
    batch * nchunks,                                    # axis-1: (batch, chunk)
    nheads,                                             # axis-2: head
)
```
Each kernel instance writes one `(BLOCK_M, BLOCK_N)` tile of `Y[b, c*Q+m*BM : c*Q+(m+1)*BM, h, n*BN:(n+1)*BN]`.

### GQA pointer arithmetic
CB and C are indexed with `pid_h // ratio` (group index) rather than `pid_h` (head index):
```triton
cb_ptr += ... + (pid_h // nheads_ngroups_ratio) * stride_cb_head
C_ptr  += ... + (pid_h // nheads_ngroups_ratio) * stride_C_head
```

### `dstate` loop (`BLOCK_SIZE_DSTATE <= 128`)
When dstate fits in one tile, Y_off is computed in a single `tl.dot(C, prev_states)`.
For dstate > 128, it tiles over dstate with an inner loop.

### Inner K-loop (Y_diag) with `IS_CAUSAL=True`
```triton
K_MAX = min((pid_m + 1) * BLOCK_M, chunk_size)   # causal: only iterate up to row m
for k in range(0, K_MAX, BLOCK_K):
    cb  = load CB[m, k..k+BK]                     # (BM, BK) float32
    dA_k = load dA_cs[k..k+BK]                    # (BK,)
    # intra-chunk decay from position k to m:
    cb *= exp(min(dA_cs_m[:, None] - dA_k[None, :], 0))  # (BM, BK)
    cb *= dt_k                                            # (BK,)
    if IS_CAUSAL: apply lower-triangular mask             # zero out k > m
    x  = load x[k..k+BK, n..n+BN]                # (BK, BN) bfloat16
    acc += tl.dot(cb, x)                          # (BM, BN), fp32 accumulation
```
The `K_MAX` early exit avoids loading x for k tiles completely above the diagonal.

In [2]:
# Triton kernel (condensed annotated reference)
# Source: mamba/mamba_ssm/ops/triton/ssd_chunk_scan.py  lines 49–180
#
# @triton.autotune(configs=[BM={32..128}, BN={32..256}, BK={32..64}], key=['chunk_size','hdim','dstate'])
# @triton.jit
# def _chunk_scan_fwd_kernel(
#   cb_ptr,         # (batch, nchunks, ngroups, chunk_size, chunk_size)  f32
#   x_ptr,          # (batch, seqlen, nheads, hdim)                      bf16
#   z_ptr,          # (batch, seqlen, nheads, hdim)  or None             bf16
#   out_ptr,        # (batch, seqlen, nheads, hdim)                      bf16  [output]
#   out_x_ptr,      # (batch, seqlen, nheads, hdim)  or None             bf16  [pre-z output]
#   dt_ptr,         # (batch, nheads, nchunks, chunk_size)               f32
#   dA_cumsum_ptr,  # (batch, nheads, nchunks, chunk_size)               f32
#   seq_idx_ptr,    # (batch, seqlen)                or None             i32
#   C_ptr,          # (batch, seqlen, ngroups, dstate)                   bf16
#   prev_states_ptr,# (batch, nchunks, nheads, hdim, dstate)             f32
#   D_ptr,          # (nheads,) or (nheads, hdim)    or None             f32
# ):
#   # ── Decode grid indices ─────────────────────────────────────────────────
#   pid_bc = tl.program_id(axis=1)                      # encodes (batch, chunk)
#   pid_c  = pid_bc // batch
#   pid_b  = pid_bc - pid_c * batch
#   pid_h  = tl.program_id(axis=2)                      # head index
#   num_pid_n = tl.cdiv(hdim, BLOCK_SIZE_N)
#   pid_m  = tl.program_id(axis=0) // num_pid_n         # row tile in chunk_size
#   pid_n  = tl.program_id(axis=0) % num_pid_n          # col tile in hdim
#
#   # ── Advance pointers to this (batch, chunk, head) ──────────────────────
#   # GQA: CB and C use group index (pid_h // ratio) instead of pid_h
#   cb_ptr += pid_b*... + pid_c*chunk_size*... + (pid_h // ratio)*stride_cb_head
#   C_ptr  += pid_b*... + pid_c*chunk_size*... + (pid_h // ratio)*stride_C_head
#   x_ptr  += pid_b*... + pid_c*chunk_size*stride_x_seqlen + pid_h*stride_x_head
#   prev_states_ptr += pid_b*... + pid_c*stride_states_chunk + pid_h*stride_states_head
#
#   offs_m = pid_m * BLOCK_M + arange(0, BLOCK_M)       # row positions in chunk
#   offs_n = pid_n * BLOCK_N + arange(0, BLOCK_N)       # col positions in hdim
#   dA_cs_m = load(dA_cumsum + offs_m)                  # (BM,) cumulative decay at rows
#   acc = zeros((BLOCK_M, BLOCK_N), f32)
#
#   # ══ Y_off: inter-chunk contribution ════════════════════════════════════
#   # C[m, :] @ states[:, n] * exp(dA_cs_m)
#   # Shapes: (BM, dstate) @ (dstate, BN) → (BM, BN)
#   scale_m = exp(dA_cs_m)                              # (BM,)  or 0 for cross-doc
#   C       = load C_ptrs  # (BM, dstate) bf16
#   states  = load prev_states_ptrs  # (dstate, BN) f32
#   acc    += tl.dot(C, states.to(bf16)) * scale_m[:, None]
#   # (for dstate > 128: loop over dstate in BLOCK_SIZE_K chunks)
#
#   # ══ Y_diag: intra-chunk causal scan ════════════════════════════════════
#   K_MAX = min((pid_m + 1) * BLOCK_M, chunk_size)      # causal early exit
#   for k in range(0, K_MAX, BLOCK_K):
#       cb   = load CB[offs_m, k + offs_k]              # (BM, BK) f32
#       dA_k = load dA_cumsum[k + offs_k]               # (BK,)
#       dt_k = load dt[k + offs_k]                      # (BK,)
#       # intra-chunk decay: exp(min(dA_m[:, None] - dA_k[None, :], 0))
#       cb  *= exp(min(dA_cs_m[:, None] - dA_k[None, :], 0))  # (BM, BK)
#       cb  *= dt_k[None, :]                            # scale by ZOH step
#       if IS_CAUSAL: cb = where(offs_m[:,None] >= k+offs_k[None,:], cb, 0)
#       x    = load x[k + offs_k, offs_n]              # (BK, BN) bf16
#       acc += tl.dot(cb.to(bf16), x)                  # (BM, BN) fp32 accumulation
#
#   # ══ Optional: D residual, z gating ════════════════════════════════════
#   # if HAS_D: acc += x * D
#   # if HAS_Z: acc *= z * sigmoid(z)  [SiLU gating]
#
#   store(out_ptr + offs_m[:, None] * stride_seqlen + offs_n[None, :], acc)

print("Triton kernel annotated above")

Triton kernel annotated above


## Naive JAX

Uses `jnp.matmul` for the two GEMM terms. No custom kernel, no seq_idx, no D/z.

For `Y_diag` we use pre-computed `CB` (same as what Triton takes as input) rather than
re-deriving `C @ B^T` — this keeps the comparison apples-to-apples for the scan step.
The intra-chunk decay matrix `L[b,h,c,m,k] = exp(min(dA[m]-dA[k], 0)) * dt[k] * causal_mask[m,k]`
is computed explicitly and applied to `CB` before the matmul.

In [3]:
def chunk_scan_naive(CB, x, dt, dA_cumsum, C, states):
    """
    Minimal JAX implementation of _chunk_scan_fwd.

    CB:         (batch, nchunks, ngroups, chunk_size, chunk_size)  float32
    x:          (batch, seqlen, nheads, hdim)                      bfloat16
    dt:         (batch, nheads, nchunks, chunk_size)               float32
    dA_cumsum:  (batch, nheads, nchunks, chunk_size)               float32
    C:          (batch, seqlen, ngroups, dstate)                   bfloat16
    states:     (batch, nchunks, nheads, hdim, dstate)             float32
    Returns:    (batch, seqlen, nheads, hdim)                      bfloat16
    """
    batch, seqlen, nheads, hdim = x.shape
    _, _, nchunks, chunk_size   = dt.shape
    _, _, ngroups, dstate       = C.shape
    ratio = nheads // ngroups

    # ── Reshape into (batch, nchunks, nheads, chunk_size, ...) ──────────────
    # x_c: (B, C, H, Q, P)
    x_c = x.reshape(batch, nchunks, chunk_size, nheads, hdim).transpose(0, 1, 3, 2, 4)

    # C_c: (B, C, H, Q, N) — expand ngroups → nheads
    C_c = C.reshape(batch, nchunks, chunk_size, ngroups, dstate)
    if ratio > 1:
        C_c = jnp.repeat(C_c, ratio, axis=3)          # (B, C, Q, H, N)
    C_c = C_c.transpose(0, 1, 3, 2, 4)                 # (B, C, H, Q, N)

    # ── Y_off: inter-chunk ───────────────────────────────────────────────────
    # state_decay = exp(dA_cumsum): (B, H, C, Q) → (B, C, H, Q)
    state_decay = jnp.exp(dA_cumsum).transpose(0, 2, 1, 3)   # (B, C, H, Q)
    # C_scaled[b,c,h,m,n] = C[b,c,h,m,n] * exp(dA[b,h,c,m])
    C_scaled = C_c.astype(jnp.float32) * state_decay[..., None]  # (B, C, H, Q, N)
    # Y_off = C_scaled @ states^T:  (B,C,H,Q,N) @ (B,C,H,N,P) → (B,C,H,Q,P)
    Y_off = jnp.matmul(C_scaled, states.transpose(0, 1, 2, 4, 3).astype(jnp.float32))

    # ── Y_diag: intra-chunk causal scan ─────────────────────────────────────
    # Intra-chunk decay: L[b,h,c,m,k] = exp(min(dA[m]-dA[k], 0))  (B,H,C,Q,Q)
    dA_m = dA_cumsum[..., :, None]   # (B, H, C, Q, 1)
    dA_k = dA_cumsum[..., None, :]   # (B, H, C, 1, Q)
    L = jnp.exp(jnp.minimum(dA_m - dA_k, 0.0))                   # (B, H, C, Q, Q)
    # Causal mask (lower triangular, m >= k)
    causal = jnp.tril(jnp.ones((chunk_size, chunk_size), dtype=bool))
    L = jnp.where(causal, L, 0.0)                                  # (B, H, C, Q, Q)
    # Scale by dt: dt[b,h,c,k] multiplies along the k (column) dimension
    L = L * dt[..., None, :]           # (B, H, C, Q, Q):  [..., m, k] * dt[..., k]

    # Expand CB from ngroups → nheads and apply L
    CB_h = jnp.repeat(CB, ratio, axis=2) if ratio > 1 else CB     # (B, C, H, Q, Q)
    L_t  = L.transpose(0, 2, 1, 3, 4)                             # (B, C, H, Q, Q)
    CB_scaled = CB_h.astype(jnp.float32) * L_t                    # (B, C, H, Q, Q)

    # Y_diag = CB_scaled @ x_c:  (B,C,H,Q,Q) @ (B,C,H,Q,P) → (B,C,H,Q,P)
    Y_diag = jnp.matmul(CB_scaled, x_c.astype(jnp.float32))

    # ── Combine and reshape ──────────────────────────────────────────────────
    # (B,C,H,Q,P) → transpose → (B,C,Q,H,P) → reshape → (B,L,H,P)
    Y = (Y_diag + Y_off).transpose(0, 1, 3, 2, 4).reshape(batch, seqlen, nheads, hdim)
    return Y.astype(jnp.bfloat16)

print("chunk_scan_naive defined")

chunk_scan_naive defined


## Pallas Kernel Design

### Grid: `(BCH, PM, PN)`
```
BCH = batch * nchunks * nheads     — one slot per (batch, chunk, head)
PM  = chunk_size // BLOCK_M        — row tiles of the output Q-dimension
PN  = hdim // BLOCK_N              — col tiles of the output P-dimension
```
Each instance writes one `(BLOCK_M, BLOCK_N)` tile.

### Pre-flatten reshapes (wrapper)

| Tensor | Original shape | Flat shape | Reshape |
|--------|----------------|------------|---------|
| `x`    | `(B,L,H,P)` | `(BCH,Q,P)` | reshape+transpose |
| `CB`   | `(B,C,G,Q,Q)` | `(BCG,Q,Q)` | simple reshape |
| `C`    | `(B,L,G,N)` | `(BCG,Q,N)` | reshape+transpose |
| `dt`   | `(B,H,C,Q)` | `(BCH,Q)` | transpose+reshape |
| `dA`   | `(B,H,C,Q)` | `(BCH,Q)` | transpose+reshape |
| `states` | `(B,C,H,P,N)` | `(BCH,P,N)` | simple reshape |
| `out`  | `(B,L,H,P)` | `(BCH,Q,P)` | same as x |

where `BCG = batch * nchunks * ngroups`, `Q = chunk_size`.

### BlockSpecs

| Ref | Block shape | Index map | Pattern |
|-----|------------|-----------|--------|
| `cb`     | `(1, BM, Q)` | `(bcg, pm, 0)` | load row tile + all K cols |
| `C`      | `(1, BM, N)` | `(bcg, pm, 0)` | row tile, full dstate |
| `x`      | `(1, Q, BN)` | `(bch, 0, pn)` | full chunk rows, col tile |
| `dt`     | `(1, Q)`     | `(bch, 0)`     | full chunk dt |
| `dA_m`   | `(1, BM)`    | `(bch, pm)`    | just the BM m-rows of dA |
| `dA`     | `(1, Q)`     | `(bch, 0)`     | full chunk dA (for decay) |
| `states` | `(1, BN, N)` | `(bch, pn, 0)` | col tile, full dstate |
| `out`    | `(1, BM, BN)`| `(bch, pm, pn)`| output tile |

`dA` is passed **twice**: once as `(1, BM)` for `scale_m = exp(dA_m)` (Y_off),
once as `(1, Q)` for the full decay matrix (Y_diag).  
GQA mapping: `bcg = (bch // nheads) * ngroups + (bch % nheads) // ratio`.

### Why no inner K-loop?

Loading the full `(BLOCK_M, chunk_size)` CB and `(chunk_size, BLOCK_N)` x at once
enables a single `jnp.matmul` for Y_diag — no sequential K-tiling needed.
For `chunk_size ≤ 256` and `BLOCK_M ≤ 64` the tiles fit comfortably in registers.

Pallas has no `num_stages` software pipelining primitive, so a K-loop would NOT help
with latency hiding — the single large matmul is strictly better.

The causal mask `m_abs[:, None] >= k_abs[None, :]` is applied to `CB` before the matmul.
`m_abs = pl.program_id(1) * BLOCK_M + arange(BLOCK_M)` — arithmetic on a traced int
is fine in JAX; only *indexing* with a traced int fails.

In [4]:
def _make_chunk_scan_kernel(BLOCK_M, BLOCK_N, chunk_size):
    """
    Factory that captures tile sizes and chunk_size as compile-time constants.

    Refs (in order):
      cb_ref     (1, BLOCK_M, chunk_size)  f32   — row tile of CB, full K cols
      C_ref      (1, BLOCK_M, dstate)      bf16  — row tile of C
      x_ref      (1, chunk_size, BLOCK_N)  bf16  — full chunk rows, col tile
      dt_ref     (1, chunk_size)           f32   — full chunk dt
      dA_m_ref   (1, BLOCK_M)              f32   — dA_cumsum at the m-rows
      dA_ref     (1, chunk_size)           f32   — dA_cumsum full chunk (for decay)
      states_ref (1, BLOCK_N, dstate)      f32   — col tile of states
      out_ref    (1, BLOCK_M, BLOCK_N)     f32   — output tile (cast to bf16 in wrapper)
    """
    def _chunk_scan_fwd_kernel(
        cb_ref, C_ref, x_ref, dt_ref, dA_m_ref, dA_ref, states_ref, out_ref
    ):
        # ── Load all tiles ──────────────────────────────────────────────────
        # ref[0, :, :] = ref[scalar, full, full] — confirmed-safe Pallas access
        cb     = cb_ref[0, :, :]        # (BM, Q)  f32 — CB row tile, all K cols
        C      = C_ref[0, :, :]         # (BM, N)  bf16
        x      = x_ref[0, :, :]         # (Q, BN)  bf16
        dt     = dt_ref[0, :]           # (Q,)     f32
        dA_m   = dA_m_ref[0, :]         # (BM,)    f32 — cumsum at row positions
        dA     = dA_ref[0, :]           # (Q,)     f32 — cumsum at all K positions
        states = states_ref[0, :, :]    # (BN, N)  f32

        # ══ Y_off: inter-chunk contribution ════════════════════════════════
        # acc[m, p] = exp(dA_m[m]) * sum_n C[m,n] * states[p,n]
        #           = scale_m[m] * (C @ states^T)[m, p]
        # Triton: acc = tl.dot(C, prev_states.to(bf16)) * scale_m[:, None]
        scale_m = jnp.exp(dA_m)                                    # (BM,)
        acc = (
            jnp.matmul(C.astype(jnp.float32), states.astype(jnp.float32).T)
            * scale_m[:, None]
        )  # (BM, N) @ (N, BN) → (BM, BN)  f32

        # ══ Y_diag: intra-chunk causal scan ════════════════════════════════
        # decay[m, k] = exp(min(dA_m[m] - dA[k], 0))  — intra-chunk decay m→k
        # Triton: cb *= exp(tl.minimum(dA_cs_m[:,None] - dA_k[None,:], 0))
        decay = jnp.exp(
            jnp.minimum(dA_m[:, None] - dA[None, :], 0.0)
        )  # (BM, Q)  f32

        # Scale CB by decay and dt (ZOH discretization):
        # Triton: cb *= dt_k  then cast to bf16 before tl.dot
        cb_scaled = cb * decay * dt[None, :]    # (BM, Q)  f32

        # Causal mask: only positions k <= m contribute (IS_CAUSAL=True always)
        # m_abs[i] = pm * BLOCK_M + i   — absolute row position in chunk
        # Triton: K_MAX = min((pid_m+1)*BM, chunk_size) early-exits the loop;
        #         within the last tile: where(offs_m[:,None] >= k+offs_k[None,:], cb, 0)
        # Pallas: compute causal mask from program_id — arithmetic on traced int is fine
        pm    = pl.program_id(axis=1)                              # traced int32
        m_abs = pm * BLOCK_M + jnp.arange(BLOCK_M)                # (BM,) traced
        k_abs = jnp.arange(chunk_size)                             # (Q,)  static
        causal = m_abs[:, None] >= k_abs[None, :]                  # (BM, Q) bool
        cb_scaled = jnp.where(causal, cb_scaled, 0.0)              # (BM, Q)  f32

        # Y_diag = cb_scaled @ x:  (BM, Q) @ (Q, BN) → (BM, BN)
        # Triton: acc += tl.dot(cb.to(bf16), x)  — fp32 accumulation
        # Pallas: bf16 matmul unsupported on Triton backend — use f32 @ f32
        acc = acc + jnp.matmul(cb_scaled, x.astype(jnp.float32))  # (BM, BN)  f32

        # ── Write output ────────────────────────────────────────────────────
        # Output dtype is float32; wrapper casts to bfloat16 after reshape
        out_ref[0, :, :] = acc    # (BM, BN)  f32

    return _chunk_scan_fwd_kernel

print("_make_chunk_scan_kernel defined")

_make_chunk_scan_kernel defined


In [5]:
def chunk_scan_pallas(CB, x, dt, dA_cumsum, C, states, BLOCK_M=32, BLOCK_N=64):
    """
    Pallas implementation of _chunk_scan_fwd.

    CB:         (batch, nchunks, ngroups, chunk_size, chunk_size)  float32
    x:          (batch, seqlen, nheads, hdim)                      bfloat16
    dt:         (batch, nheads, nchunks, chunk_size)               float32
    dA_cumsum:  (batch, nheads, nchunks, chunk_size)               float32
    C:          (batch, seqlen, ngroups, dstate)                   bfloat16
    states:     (batch, nchunks, nheads, hdim, dstate)             float32
    Returns:    (batch, seqlen, nheads, hdim)                      bfloat16
    """
    batch, seqlen, nheads, hdim = x.shape
    _, _, nchunks, chunk_size   = dt.shape
    _, _, ngroups, dstate       = C.shape
    ratio = nheads // ngroups

    BCH = batch * nchunks * nheads    # flat (batch, chunk, head) index space
    BCG = batch * nchunks * ngroups   # flat (batch, chunk, group) index space
    PM  = chunk_size // BLOCK_M
    PN  = hdim // BLOCK_N

    # ── Pre-flatten inputs ──────────────────────────────────────────────────
    #
    # Convention: bch = b*(nchunks*nheads) + c*nheads + h
    #             bcg = b*(nchunks*ngroups) + c*ngroups + g = b*(nchunks*ngroups) + c*ngroups + h//ratio
    #
    # x: (B, L, H, P) → (B, C, Q, H, P) → (B, C, H, Q, P) → (BCH, Q, P)
    x_flat = (
        x.reshape(batch, nchunks, chunk_size, nheads, hdim)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCH, chunk_size, hdim)
    )
    # CB: (B, C, G, Q, Q) → (BCG, Q, Q)   [already contiguous in leading dims]
    CB_flat = CB.reshape(BCG, chunk_size, chunk_size)

    # C: (B, L, G, N) → (B, C, Q, G, N) → (B, C, G, Q, N) → (BCG, Q, N)
    C_flat = (
        C.reshape(batch, nchunks, chunk_size, ngroups, dstate)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCG, chunk_size, dstate)
    )
    # dt:  (B, H, C, Q) → (B, C, H, Q) → (BCH, Q)
    dt_flat  = dt.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    # dA:  (B, H, C, Q) → (BCH, Q)  (same layout as dt)
    dA_flat  = dA_cumsum.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    # states: (B, C, H, P, N) → (BCH, P, N)   [C, H are already adjacent after cumsum]
    states_flat = states.reshape(BCH, hdim, dstate)

    # ── BlockSpecs ──────────────────────────────────────────────────────────
    _nh = nheads; _ng = ngroups; _r = ratio
    _Q  = chunk_size; _N  = dstate

    # GQA index: bcg = (bch // nheads) * ngroups + (bch % nheads) // ratio
    # Mirrors Triton: (pid_h // nheads_ngroups_ratio) for cb_ptr and C_ptr
    def bcg(bch, pm, pn):
        return (bch // _nh) * _ng + (bch % _nh) // _r

    in_specs = [
        # CB_flat: (BCG, Q, Q)  block (1, BM, Q) — row tile, full K cols
        # Triton: cb_ptr += ... + (pid_h//ratio)*stride_cb_head
        pl.BlockSpec((1, BLOCK_M, _Q),  lambda bch, pm, pn: (bcg(bch, pm, pn), pm, 0)),

        # C_flat: (BCG, Q, N)  block (1, BM, N) — row tile, full dstate
        # Triton: C_ptr += ... + (pid_h//ratio)*stride_C_head
        pl.BlockSpec((1, BLOCK_M, _N),  lambda bch, pm, pn: (bcg(bch, pm, pn), pm, 0)),

        # x_flat: (BCH, Q, P)  block (1, Q, BN) — full chunk rows, col tile
        # Triton: x_ptr += ... + pid_h*stride_x_head; loads (BK, BN) in inner loop
        pl.BlockSpec((1, _Q, BLOCK_N),  lambda bch, pm, pn: (bch, 0, pn)),

        # dt_flat: (BCH, Q)  block (1, Q) — full chunk dt
        pl.BlockSpec((1, _Q),           lambda bch, pm, pn: (bch, 0)),

        # dA_m: (BCH, Q) viewed as (BCH, Q), block (1, BM) at (bch, pm)
        # Gives exactly dA_cumsum[bch, pm*BM : (pm+1)*BM] for scale_m
        # Triton: dA_cs_m = load(dA_cumsum + offs_m * stride)  (BM,)
        pl.BlockSpec((1, BLOCK_M),      lambda bch, pm, pn: (bch, pm)),

        # dA_full: same tensor, full Q — needed for decay[m,k] = exp(min(dA_m-dA_k,0))
        # Triton: dA_cs_k = load(dA_cumsum + offs_k)  loaded inside the K-loop
        pl.BlockSpec((1, _Q),           lambda bch, pm, pn: (bch, 0)),

        # states_flat: (BCH, P, N)  block (1, BN, N) — col tile, full dstate
        # Triton: prev_states_ptr += ... + pid_h*stride_states_head
        pl.BlockSpec((1, BLOCK_N, _N),  lambda bch, pm, pn: (bch, pn, 0)),
    ]
    out_specs = [
        # out: (BCH, Q, P)  block (1, BM, BN) — output tile (float32)
        pl.BlockSpec((1, BLOCK_M, BLOCK_N), lambda bch, pm, pn: (bch, pm, pn)),
    ]

    # dA_flat is passed TWICE: once sliced to BM (for scale_m), once full Q (for decay)
    out_flat, = pl.pallas_call(
        _make_chunk_scan_kernel(BLOCK_M, BLOCK_N, chunk_size),
        out_shape=[jax.ShapeDtypeStruct((BCH, chunk_size, hdim), jnp.float32)],
        in_specs=in_specs,
        out_specs=out_specs,
        grid=(BCH, PM, PN),
        compiler_params=CompilerParams(),
    )(CB_flat, C_flat, x_flat, dt_flat, dA_flat, dA_flat, states_flat)

    # ── Reshape output: (BCH, Q, P) → (B, C, H, Q, P) → (B, C, Q, H, P) → (B, L, H, P)
    out = (
        out_flat
        .reshape(batch, nchunks, nheads, chunk_size, hdim)
        .transpose(0, 1, 3, 2, 4)
        .reshape(batch, seqlen, nheads, hdim)
    )
    return out.astype(jnp.bfloat16)

print("chunk_scan_pallas defined")

chunk_scan_pallas defined


## Correctness Check

In [6]:
def make_inputs(batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size, seed=0):
    """Generate random JAX inputs matching _chunk_scan_fwd's expected shapes."""
    nchunks = seqlen // chunk_size
    ks = jax.random.split(jax.random.PRNGKey(seed), 6)
    x      = jax.random.normal(ks[0], (batch, seqlen,  nheads,  hdim),   dtype=jnp.bfloat16)
    C      = jax.random.normal(ks[1], (batch, seqlen,  ngroups, dstate), dtype=jnp.bfloat16)
    B      = jax.random.normal(ks[2], (batch, seqlen,  ngroups, dstate), dtype=jnp.bfloat16)
    dt     = jax.nn.softplus(jax.random.normal(ks[3], (batch, nheads, nchunks, chunk_size)))
    dA_cs  = -jnp.cumsum(
        jax.random.uniform(ks[4], (batch, nheads, nchunks, chunk_size)) * dt, axis=-1
    )  # negative cumulative sum → monotone decreasing (valid log-decay)
    states = jax.random.normal(ks[5], (batch, nchunks, nheads, hdim, dstate))
    # Compute CB with naive matmul (same as the benchmark's CB)
    B_c    = B.reshape(batch, nchunks, chunk_size, ngroups, dstate).transpose(0,1,3,2,4)
    C_c    = C.reshape(batch, nchunks, chunk_size, ngroups, dstate).transpose(0,1,3,2,4)
    CB     = jnp.matmul(
        C_c.astype(jnp.float32),
        B_c.transpose(0,1,2,4,3).astype(jnp.float32)
    )  # (batch, nchunks, ngroups, chunk_size, chunk_size)
    return dict(CB=CB, x=x, dt=dt, dA=dA_cs, C=C, B=B, states=states)

print("make_inputs defined")

make_inputs defined


In [7]:
batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = 2, 512, 8, 64, 64, 1, 64
inp = make_inputs(batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size)
CB, x, dt, dA, C, B, states = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

# ── Triton reference ─────────────────────────────────────────────────────────
CB_t  = to_torch_f32(CB);  x_t  = to_torch_bf16(x)
dt_t  = to_torch_f32(dt);  dA_t = to_torch_f32(dA)
C_t   = to_torch_bf16(C);  st_t = to_torch_f32(states)
Y_tri, _ = _chunk_scan_fwd(CB_t, x_t, dt_t, dA_t, C_t, st_t)
# Triton output is bfloat16; convert via float32 (BF16→FP32 is exact)
Y_ref = jnp.array(Y_tri.float().cpu().numpy()).astype(jnp.bfloat16)
print(f"Y shape: {Y_ref.shape}   |Y|_max: {float(jnp.abs(Y_ref.astype(jnp.float32)).max()):.1f}")

# ── Naive ────────────────────────────────────────────────────────────────────
Y_naive = jax.jit(lambda *a: chunk_scan_naive(*a))(CB, x, dt, dA, C, states)
diff_naive = float(jnp.max(jnp.abs(Y_naive.astype(jnp.float32) - Y_ref.astype(jnp.float32))))
print(f"Naive  vs Triton — max abs diff: {diff_naive:.3f}")

# ── Pallas ───────────────────────────────────────────────────────────────────
Y_pal = jax.jit(
    lambda *a: chunk_scan_pallas(*a, BLOCK_M=32, BLOCK_N=64)
)(CB, x, dt, dA, C, states)
diff_pal   = float(jnp.max(jnp.abs(Y_pal.astype(jnp.float32) - Y_ref.astype(jnp.float32))))
diff_np    = float(jnp.max(jnp.abs(Y_pal.astype(jnp.float32) - Y_naive.astype(jnp.float32))))
print(f"Pallas vs Triton — max abs diff: {diff_pal:.3f}")
print(f"Pallas vs Naive  — max abs diff: {diff_np:.4f}  (should be <0.5)")

# ── Notes on expected precision ──────────────────────────────────────────────
# Triton kernel casts cb_scaled to BF16 before tl.dot(cb.to(bf16), x).
# Our Pallas/naive use F32 @ F32 — strictly more accurate.
# At output magnitude 64-128, BF16 step = 0.5, so ≤1 BF16 ULP difference vs Triton is expected.
# Threshold: 2.0 = 2 BF16 steps at magnitude ≤256.
THRESHOLD = 2.0
assert diff_naive < THRESHOLD, f"Naive failed: {diff_naive}"
assert diff_pal   < THRESHOLD, f"Pallas failed: {diff_pal}"
assert diff_np    < 0.5,       f"Naive vs Pallas too large: {diff_np}"
print(f"\nAll assertions passed (threshold = {THRESHOLD})")

Y shape: (2, 512, 8, 64)   |Y|_max: 193.0
Naive  vs Triton — max abs diff: 0.500
Pallas vs Triton — max abs diff: 0.500
Pallas vs Naive  — max abs diff: 0.2500  (should be <0.5)

All assertions passed (threshold = 2.0)


In [8]:
# Test Nemotron config: ngroups=8, ratio=16
inp2 = make_inputs(1, 256, 128, 64, 128, 8, 256)
CB2, x2, dt2, dA2, C2, B2, st2 = [inp2[k] for k in ('CB','x','dt','dA','C','B','states')]

Y_ref2, _ = _chunk_scan_fwd(
    to_torch_f32(CB2), to_torch_bf16(x2), to_torch_f32(dt2),
    to_torch_f32(dA2), to_torch_bf16(C2), to_torch_f32(st2)
)
Y_ref2 = jnp.array(Y_ref2.float().cpu().numpy()).astype(jnp.bfloat16)

Y_pal2 = jax.jit(
    lambda *a: chunk_scan_pallas(*a, BLOCK_M=64, BLOCK_N=64)
)(CB2, x2, dt2, dA2, C2, st2)

Y_naive2 = jax.jit(lambda *a: chunk_scan_naive(*a))(CB2, x2, dt2, dA2, C2, st2)

diff_pal2   = float(jnp.max(jnp.abs(Y_pal2.astype(jnp.float32) - Y_ref2.astype(jnp.float32))))
diff_naive2 = float(jnp.max(jnp.abs(Y_naive2.astype(jnp.float32) - Y_ref2.astype(jnp.float32))))
diff_np2    = float(jnp.max(jnp.abs(Y_pal2.astype(jnp.float32) - Y_naive2.astype(jnp.float32))))
ymax2 = float(jnp.abs(Y_ref2.astype(jnp.float32)).max())

print(f"Nemotron (H=128, G=8, cs=256) |Y|_max: {ymax2:.1f}")
print(f"  Naive  vs Triton: {diff_naive2:.3f}")
print(f"  Pallas vs Triton: {diff_pal2:.3f}")
print(f"  Pallas vs Naive:  {diff_np2:.4f}")
assert diff_pal2 < 2.0 and diff_naive2 < 2.0 and diff_np2 < 0.5
print("  OK")

Nemotron (H=128, G=8, cs=256) |Y|_max: 276.0
  Naive  vs Triton: 1.000
  Pallas vs Triton: 1.000
  Pallas vs Naive:  0.2500
  OK


## Autotune: `(BLOCK_M, BLOCK_N)`

In [9]:
_cfg = dict(batch=1, seqlen=2048, nheads=32, hdim=64, dstate=64, ngroups=1, chunk_size=64)
_inp = make_inputs(**_cfg)
_CB, _x, _dt, _dA, _C, _st = [_inp[k] for k in ('CB','x','dt','dA','C','states')]

N_WARMUP, N_RUNS = 5, 20
BEST_BM, BEST_BN = 32, 64  # defaults; updated below
best_ms = float('inf')

print(f"Autotuning for: {_cfg}")
print(f"  {'BM':>4} {'BN':>4}     ms")
print("-" * 28)
for BM in [16, 32, 64]:
    for BN in [16, 32, 64]:
        if _cfg['chunk_size'] % BM != 0 or _cfg['hdim'] % BN != 0:
            continue
        fn = jax.jit(lambda cb,x,dt,da,c,st: chunk_scan_pallas(cb,x,dt,da,c,st,BLOCK_M=BM,BLOCK_N=BN))
        for _ in range(N_WARMUP): fn(_CB,_x,_dt,_dA,_C,_st).block_until_ready()
        t0 = time.perf_counter()
        for _ in range(N_RUNS): fn(_CB,_x,_dt,_dA,_C,_st).block_until_ready()
        ms = (time.perf_counter()-t0)/N_RUNS*1e3
        tag = "  <-- best" if ms < best_ms else ""
        print(f"  {BM:4d} {BN:4d}  {ms:6.3f} ms{tag}")
        if ms < best_ms:
            best_ms, BEST_BM, BEST_BN = ms, BM, BN

print(f"\nBest: BLOCK_M={BEST_BM}, BLOCK_N={BEST_BN}")

Autotuning for: {'batch': 1, 'seqlen': 2048, 'nheads': 32, 'hdim': 64, 'dstate': 64, 'ngroups': 1, 'chunk_size': 64}
    BM   BN     ms
----------------------------
    16   16   1.354 ms  <-- best
    16   32   1.241 ms  <-- best
    16   64   1.254 ms
    32   16   1.232 ms  <-- best
    32   32   1.310 ms
    32   64   1.811 ms
    64   16   2.576 ms
    64   32   1.887 ms
    64   64   1.797 ms

Best: BLOCK_M=32, BLOCK_N=16


## Benchmarks

**Scan-only**: CB is pre-computed and not counted — isolates the scan kernel.

**Full pipeline**: includes CB computation (naive matmul) for Pallas;
for Triton, uses Triton's `_bmm_chunk_fwd` (which is faster than naive for some configs).

All timings use `fori_loop` amortization to eliminate JAX Python dispatch overhead (~1 ms).

In [10]:
SHAPE_SWEEP = [
    # (batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size),   label
    ((1, 2048, 32,  64,  64, 1,  64),  "Mamba2 standard   B=1 H=32  P=64  N=64  Q=64 "),
    ((1, 2048, 32,  64,  64, 1, 256),  "Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256"),
    ((1, 2048, 128, 64, 128, 8, 256),  "Nemotron          B=1 H=128 P=64  N=128 Q=256"),
    ((2, 2048, 32,  64,  64, 1,  64),  "Batched           B=2 H=32  P=64  N=64  Q=64 "),
]

N_AMORTIZE = 200

def _best_block(chunk_size, hdim):
    """Return best (BM, BN) from autotune result, fallback to largest valid."""
    if chunk_size % BEST_BM == 0 and hdim % BEST_BN == 0:
        return BEST_BM, BEST_BN
    bm = max(b for b in [16,32,64] if chunk_size % b == 0)
    bn = max(b for b in [16,32,64] if hdim % b == 0)
    return bm, bn

def bmm_naive_jax(C, B, chunk_size):
    """Compute CB with naive jnp.matmul (no custom kernel)."""
    batch, seqlen, ngroups, dstate = C.shape
    nchunks = seqlen // chunk_size
    C_c = C.reshape(batch,nchunks,chunk_size,ngroups,dstate).transpose(0,1,3,2,4)
    B_c = B.reshape(batch,nchunks,chunk_size,ngroups,dstate).transpose(0,1,3,2,4)
    return jnp.matmul(C_c.astype(jnp.float32), B_c.transpose(0,1,2,4,3).astype(jnp.float32))

def make_amortized(fn, n):
    @jax.jit
    def inner(*args):
        out0 = jnp.zeros_like(fn(*args))
        return lax.fori_loop(0, n, lambda i, _: fn(*args), out0)
    return inner

def bench_amortized(fn_am, args, n, warmup=3, runs=5):
    fn_am(*args).block_until_ready()
    for _ in range(warmup): fn_am(*args).block_until_ready()
    t0 = time.perf_counter()
    for _ in range(runs): fn_am(*args).block_until_ready()
    return (time.perf_counter()-t0)/runs/n*1e3

def bench_triton_amortized(fn_t, args_t, n, warmup=10, runs=5):
    """Triton amortized: plain Python loop (Triton has low dispatch overhead)."""
    for _ in range(warmup): fn_t(*args_t)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n*runs): fn_t(*args_t)
    torch.cuda.synchronize()
    return (time.perf_counter()-t0)/runs/n*1e3

print("Benchmark utilities defined")

Benchmark utilities defined


In [11]:
print("═" * 82)
print("SCAN-ONLY benchmark  (CB pre-computed, not counted)")
print(f"{'Config':<45} {'Triton':>8} {'Naive':>8} {'Pallas':>8}")
print(f"{'':45} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8}")
print("─" * 82)

for cfg, label in SHAPE_SWEEP:
    batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = cfg
    inp = make_inputs(*cfg)
    CB, x, dt, dA, C, B, st = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

    # Triton: _chunk_scan_fwd (CB already float32 torch tensor)
    CB_t = to_torch_f32(CB); x_t = to_torch_bf16(x)
    dt_t = to_torch_f32(dt); dA_t = to_torch_f32(dA)
    C_t  = to_torch_bf16(C); st_t = to_torch_f32(st)
    t_tri = bench_triton_amortized(
        lambda: _chunk_scan_fwd(CB_t, x_t, dt_t, dA_t, C_t, st_t), (), N_AMORTIZE
    )

    # Naive JAX scan
    fn_naive_am = make_amortized(lambda cb,xj,dtj,daj,cj,sj: chunk_scan_naive(cb,xj,dtj,daj,cj,sj), N_AMORTIZE)
    t_naive = bench_amortized(fn_naive_am, (CB,x,dt,dA,C,st), N_AMORTIZE)

    # Pallas scan
    BM, BN = _best_block(chunk_size, hdim)
    fn_pal_am = make_amortized(lambda cb,xj,dtj,daj,cj,sj: chunk_scan_pallas(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM,BLOCK_N=BN), N_AMORTIZE)
    t_pal = bench_amortized(fn_pal_am, (CB,x,dt,dA,C,st), N_AMORTIZE)

    print(f"{label:<45} {t_tri:>8.3f} {t_naive:>8.3f} {t_pal:>8.3f}")

print("═" * 82)

══════════════════════════════════════════════════════════════════════════════════
SCAN-ONLY benchmark  (CB pre-computed, not counted)
Config                                          Triton    Naive   Pallas
                                                  (ms)     (ms)     (ms)
──────────────────────────────────────────────────────────────────────────────────
Mamba2 standard   B=1 H=32  P=64  N=64  Q=64     0.131    0.093    0.062
Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256    0.135    0.197    0.146
Nemotron          B=1 H=128 P=64  N=128 Q=256    0.180    1.272    0.748
Batched           B=2 H=32  P=64  N=64  Q=64     0.080    0.313    0.135
══════════════════════════════════════════════════════════════════════════════════


In [12]:
print("═" * 82)
print(f"FULL PIPELINE  (CB included; Triton=_bmm+_scan, Pallas/Naive=naive_bmm+scan)")
print(f"{'Config':<45} {'Triton':>8} {'Naive':>8} {'Pallas':>8}")
print(f"{'':45} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8}")
print("─" * 82)

for cfg, label in SHAPE_SWEEP:
    batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = cfg
    inp = make_inputs(*cfg)
    x, dt, dA, C, B, st = [inp[k] for k in ('x','dt','dA','C','B','states')]

    # Triton full: _bmm_chunk_fwd + _chunk_scan_fwd
    x_t  = to_torch_bf16(x);  C_t  = to_torch_bf16(C);  B_t  = to_torch_bf16(B)
    dt_t = to_torch_f32(dt);  dA_t = to_torch_f32(dA);  st_t = to_torch_f32(st)
    def triton_full():
        cb_t = _bmm_chunk_fwd(C_t, B_t, chunk_size, output_dtype=torch.float32)
        return _chunk_scan_fwd(cb_t, x_t, dt_t, dA_t, C_t, st_t)
    t_tri = bench_triton_amortized(triton_full, (), N_AMORTIZE)

    # Naive full: naive bmm + naive scan
    def naive_full(xj, dtj, daj, cj, bj, sj):
        cb = bmm_naive_jax(cj, bj, chunk_size)
        return chunk_scan_naive(cb, xj, dtj, daj, cj, sj)
    fn_naive_full = make_amortized(naive_full, N_AMORTIZE)
    t_naive = bench_amortized(fn_naive_full, (x,dt,dA,C,B,st), N_AMORTIZE)

    # Pallas full: naive bmm + pallas scan
    BM, BN = _best_block(chunk_size, hdim)
    def pallas_full(xj, dtj, daj, cj, bj, sj):
        cb = bmm_naive_jax(cj, bj, chunk_size)
        return chunk_scan_pallas(cb, xj, dtj, daj, cj, sj, BLOCK_M=BM, BLOCK_N=BN)
    fn_pal_full = make_amortized(pallas_full, N_AMORTIZE)
    t_pal = bench_amortized(fn_pal_full, (x,dt,dA,C,B,st), N_AMORTIZE)

    print(f"{label:<45} {t_tri:>8.3f} {t_naive:>8.3f} {t_pal:>8.3f}")

print("═" * 82)

══════════════════════════════════════════════════════════════════════════════════
FULL PIPELINE  (CB included; Triton=_bmm+_scan, Pallas/Naive=naive_bmm+scan)
Config                                          Triton    Naive   Pallas
                                                  (ms)     (ms)     (ms)
──────────────────────────────────────────────────────────────────────────────────
Mamba2 standard   B=1 H=32  P=64  N=64  Q=64     0.140    0.093    0.062
Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256    0.127    0.196    0.153
Nemotron          B=1 H=128 P=64  N=128 Q=256    0.207    1.481    0.850
Batched           B=2 H=32  P=64  N=64  Q=64     0.118    0.315    0.132
══════════════════════════════════════════════════════════════════════════════════


## Analysis

### Precision vs Triton

The Pallas and naive implementations compute `cb_scaled @ x` in **float32 × float32**,
while Triton does `tl.dot(cb.to(bf16), x)` — casting `cb_scaled` to BF16 first.
This produces ≤1 BF16 ULP difference at typical output magnitudes (~64-128):
at those values, BF16 step = 0.5, so max abs diff ≈ 0.5–1.0 vs Triton.

Our F32 matmul is **more accurate** than Triton in this dimension.

### What dominates the runtime

The scan kernel has two GEMM terms:

| Term | Shape | Notes |
|------|-------|-------|
| Y_off: `C @ states^T` | `(BM, dstate) @ (dstate, BN)` | small dstate (64–128) |
| Y_diag: `cb_scaled @ x` | `(BM, chunk_size) @ (chunk_size, BN)` | loads full Q rows of x |

The Y_diag GEMM dominates: it loads `(chunk_size × BN)` of x for **every** (bch, pm)
kernel instance, even though the causal mask zeros the upper-triangular entries.

### Triton's structural advantage: `K_MAX` early-exit

In Triton: `K_MAX = min((pid_m+1)*BM, chunk_size)` cuts the K-loop at the diagonal.
For pm=0: only 1 BK tile of x is loaded.  
For pm=PM-1: all BK tiles are loaded (full x).  
On average: 50% less x bandwidth for Y_diag.

In Pallas: `pm = pl.program_id(axis=1)` is a **traced** int32. We cannot use it in
`if pm * BM < chunk_size - BM:` — Python `if` on a traced value would require `lax.cond`.
Instead we load full chunk_size x and apply `jnp.where(causal, cb_scaled, 0.0)`.
The zero-multiply ops are cheap but x is still loaded — 2× bandwidth vs Triton on average.

**Impact**: most visible for small chunk_size (64, where only half tiles are fully below
the diagonal). For large chunk_size (256), most pm tiles are already below-diagonal,
so the cost is amortized.

### Triton vs Naive vs Pallas

- **Triton** wins: K_MAX causal early-exit, software pipelining (num_stages=3–4),
  hardware-tuned GEMM tile sizes across 11 autotune configs, BF16 matmul throughput.
- **Naive** materializes `L (B,H,C,Q,Q)` and `CB_h` in DRAM when ratio>1 (GQA),
  but cuBLAS saturates for large shapes.
- **Pallas** avoids intermediate materialization (decay applied in-register),
  uses GQA BlockSpec index_map (no repeat), but pays ~2× x bandwidth vs Triton.

### GQA advantage in Pallas

For ngroups=8, nheads=128 (ratio=16): naive uses `jnp.repeat(CB, 16, axis=2)` which
materializes a 16× expanded tensor in DRAM before the matmul. Pallas accesses CB
directly via the `bcg` index map — no expansion, no extra memory bandwidth.